# Data Exploration

This notebook explores the `ogbl-ddi` dataset. Each node is a drug. Each edge is a known interaction between two drugs.

Run `pip install -e .` once from the project root before using this notebook.
The first run downloads the dataset to `datasets/` (but you can change it in the next cell if you want).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch

from ddi_gnn.data import load_ddi
from ddi_gnn.plotting import plot_graph

DATA_ROOT = "../datasets"

data = load_ddi(root=DATA_ROOT)

## Size and density

In [ ]:
all_edges = torch.cat([pairs.pos for pairs in data.splits.values()])
n_edges = all_edges.shape[0]
print(f"The graph contains {data.n_nodes} drugs and {n_edges} interactions.")

n_possible = data.n_nodes * (data.n_nodes - 1) // 2
print(f"There are {n_possible} possible pairs, so the graph density is {n_edges / n_possible:.1%}.")

## Degree distribution

The degree of a drug is the number of other drugs it interacts with.

In [ ]:
degrees = np.bincount(all_edges.flatten().numpy(), minlength=data.n_nodes)

sns.histplot(degrees, kde=True)
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.title("Degree Distribution")
plt.show()

## Splits

In [ ]:
for name, pairs in data.splits.items():
    print(f"{name:>5}: {len(pairs.pos):>9} positive pairs, {len(pairs.neg):>9} negative pairs")

In [ ]:
# Does every validation and test drug also appear in the training edges?
train_nodes = set(data.splits["train"].pos.flatten().tolist())
for name in ("valid", "test"):
    nodes = set(data.splits[name].pos.flatten().tolist())
    print(f"All {name} drugs are in the training edges: {nodes <= train_nodes}")

## Drug names

In [ ]:
print(f"Drugs without a name: {data.annotation['drug_name'].isna().sum()}")

top_drugs = data.annotation.assign(degree=degrees[data.annotation["node_id"]])
top_drugs.nlargest(10, "degree")[["node_id", "drug_name", "degree"]]

## Visualizing a subgraph

50 randomly chosen drugs and the training interactions between them.

In [ ]:
generator = torch.Generator().manual_seed(42)
small = load_ddi(root=DATA_ROOT, node_limit=50, generator=generator)
print(f"The subgraph has {small.graph.edge_index.shape[1] // 2} interactions.")

plot_graph(small, figsize=(12, 12))
plt.show()